# Module 13: Recursive CTEs and Statistical SQL
**ALY 6420 | Recursive queries, multidimensional aggregation, distribution functions, and statistical interpretation**

*Course Lecture Notes*

## Module 13
### From query performance to analytical depth
Module 12 focused on measuring and optimizing SQL. This week we use iteration to generate missing calendar periods and traverse hierarchies, summarize sales at multiple levels, and investigate distributions and relationships without moving raw data out of PostgreSQL.

## Learning objectives
- Construct bounded recursive CTEs for sequences and hierarchies.
- Apply ROLLUP, CUBE, GROUPING SETS and GROUPING.
- Interpret PERCENT_RANK, CUME_DIST and NTILE, including ties.
- Compute and explain sample/population dispersion, correlation and simple regression.
- Decide when SQL summaries suffice and when Python or R is appropriate.

Database: See the Neon-hosted `sqlda` database for applied examples. The synthetic data below is generated for practice purposes.

```sql
SELECT current_database(), current_schema(), version();
SELECT table_schema, table_name, column_name, data_type
FROM information_schema.columns
WHERE table_name IN ('customers','sales','products')
ORDER BY table_schema, table_name, ordinal_position;
```

Run setup once in the same DBeaver connection. Eight customers include one nonbuyer; eight transactions span January, March and April. A genuine NULL channel and one NULL MSRP make missingness visible. Amounts are synthetic dollars, not observed ZoomZoom results.

```sql
CREATE TEMP TABLE m13_customers (customer_id int PRIMARY KEY, region text);
INSERT INTO m13_customers VALUES (1,'East'),(2,'East'),(3,'West'),(4,'West'),(5,'East'),(6,'West'),(7,'East'),(8,'West');
CREATE TEMP TABLE m13_products (product_id int PRIMARY KEY, product_type text, base_msrp numeric);
INSERT INTO m13_products VALUES (1,'Scooter',100),(2,'Scooter',200),(3,'Auto',300),(4,'Auto',400),(5,'Auto',NULL);
CREATE TEMP TABLE m13_sales (sale_id int PRIMARY KEY, customer_id int, product_id int, sale_date date, channel text, amount numeric);
INSERT INTO m13_sales VALUES
(1,1,1,'2024-01-05','Web',90),(2,1,2,'2024-01-20','Store',180),
(3,2,3,'2024-03-01','Web',270),(4,3,4,'2024-03-15','Store',360),
(5,4,1,'2024-03-20',NULL,100),(6,5,2,'2024-04-01','Web',200),
(7,6,3,'2024-04-12','Store',300),(8,7,5,'2024-04-20','Web',150);
```

```sql
SELECT COUNT(*) AS transactions, SUM(amount) AS revenue, COUNT(DISTINCT customer_id) AS buyers FROM m13_sales;
```

Expected: 8 transactions, $1,650 revenue, 7 buyers. Grain: one row per synthetic transaction. Product attributes are at product grain.

# Part 1: Recursive CTEs - WITH RECURSIVE
## 1.1 Anchor, recursive member and termination
The anchor establishes initial rows; the recursive member transforms the current working rows. `UNION ALL` retains duplicates; `UNION` removes duplicate complete rows. Recursion ends when the next working set is empty. Always design a finite traversal.

```sql
WITH RECURSIVE counter(n) AS (
    VALUES (1)
    UNION ALL
    SELECT n+1 FROM counter WHERE n < 5
)
SELECT n FROM counter ORDER BY n;
```

| Iteration | Working rows | Accumulated rows |
|---|---|---|
| Anchor | 1 | 1 |
| 1 | 2 | 1,2 |
| 2 | 3 | 1,2,3 |
| 3 | 4 | 1,2,3,4 |
| 4 | 5 | 1,2,3,4,5 |
| 5 | empty | 1,2,3,4,5 |

The recursive term sees the current working set, not every previous row. Explicit ORDER BY controls presentation, not traversal execution. [PostgreSQL §7.8](https://www.postgresql.org/docs/16/queries-with.html).

## 1.2 Gap-free monthly reporting
Ordinary aggregation omits months without transactions. Construct a calendar spine and attach facts to it.

```sql
SELECT date_trunc('month',sale_date)::date AS month_start, SUM(amount)
FROM m13_sales GROUP BY 1 ORDER BY 1;
```

```sql
WITH RECURSIVE months(month_start) AS (
    SELECT DATE '2024-01-01'
    UNION ALL
    SELECT (month_start + INTERVAL '1 month')::date
    FROM months WHERE month_start < DATE '2024-04-01'
)
SELECT m.month_start, COUNT(s.sale_id) AS transactions,
       COALESCE(SUM(s.amount),0) AS revenue
FROM months m LEFT JOIN m13_sales s
 ON s.sale_date >= m.month_start
 AND s.sale_date < m.month_start + INTERVAL '1 month'
GROUP BY m.month_start ORDER BY m.month_start;
```

| Month | Transactions | Revenue |
|---|---:|---:|
| January | 2 | 270 |
| February | 0 | 0 |
| March | 3 | 730 |
| April | 3 | 650 |

COUNT(*) would count the unmatched calendar row as one. Use a non-null fact key. The half-open date range avoids wrapping the fact date in a function, connecting to Module 12. Keep fact filters in ON or a prefiltered CTE to preserve empty months.

```sql
SELECT generate_series(TIMESTAMP '2024-01-01', TIMESTAMP '2024-04-01', INTERVAL '1 month')::date AS month_start;
```

PostgreSQL generate_series is simpler for fixed-step calendars. Recursion is especially useful when the next state depends on the previous state or data relationships.

13.1: Compound inventory forecast

**Goal and context.** Each next state depends on the previous quantity; termination is explicit.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
WITH RECURSIVE forecast(step,qty) AS (VALUES (0,100::numeric) UNION ALL SELECT step+1,qty*0.9 FROM forecast WHERE step<3) SELECT step,round(qty,1) FROM forecast ORDER BY step;
```

**Expected result:** 100, 90, 81, 72.9.

**Derived practice:** Use factor 0.8. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

100,80,64,51.2.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

## 1.3 Traversing a hierarchy
An adjacency list stores one parent ID per node. The example models a synthetic dealership organization.

```sql
CREATE TEMP TABLE m13_org (id int PRIMARY KEY, parent_id int, name text);
INSERT INTO m13_org VALUES (1,NULL,'Director'),(2,1,'East manager'),(3,1,'West manager'),(4,2,'East analyst'),(5,3,'West analyst'),(6,4,'Intern');
```

```sql
WITH RECURSIVE org_tree AS (
 SELECT id,parent_id,name,0 AS depth,ARRAY[id] AS path
 FROM m13_org WHERE id=1
 UNION ALL
 SELECT e.id,e.parent_id,e.name,t.depth+1,t.path || e.id
 FROM org_tree t JOIN m13_org e ON e.parent_id=t.id
 WHERE t.depth < 20 AND NOT (e.id=ANY(t.path))
)
SELECT id,repeat('  ',depth)||name AS hierarchy,depth,path
FROM org_tree ORDER BY path;
```

Expected traversal IDs: 1,2,4,6,3,5; maximum depth 3. A leaf with no children naturally terminates. A depth cap may truncate legitimate data; report if the cap is reached. Path tracking prevents revisiting an ID on the same path, not duplicate nodes reached through different paths.

## 1.4 Guardrails
Use a finite boundary or naturally exhausting join, cycle checks for graphs, and a sensible depth cap. UNION alone does not prevent cycles when depth or path changes. PostgreSQL also supports CYCLE syntax. An outer LIMIT can stop demand-driven evaluation in simple cases, but sorts and joins may require the full result; never rely on it as a safety mechanism.

```sql
SET statement_timeout = '10s';
-- Run bounded recursive examples here.
RESET statement_timeout;
```

## Check your understanding
What happens if the counter condition changes to n <= 5? Why does February disappear from ordinary GROUP BY? Why can a path array stop a cycle when UNION does not?

**Check:** 6 is generated; no fact rows exist in February; a changing path makes full rows distinct.

### Example 13.2: Detect a cyclic dependency

**Goal and context.** Path membership prevents infinite cycling.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
WITH RECURSIVE edges(a,b) AS (VALUES (1,2),(2,3),(3,1)), walk(node,path) AS (VALUES (1,ARRAY[1]) UNION ALL SELECT e.b,w.path||e.b FROM walk w JOIN edges e ON e.a=w.node WHERE NOT e.b=ANY(w.path)) SELECT * FROM walk ORDER BY cardinality(path);
```

**Expected result:** paths {1}, {1,2}, {1,2,3}.

**Derived practice:** Add edge (2,4). Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

A fourth path {1,2,4} appears.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

# Part 2: Multidimensional aggregation
## 2.1 From GROUP BY to grouping sets
Chapter 8 (pp. 158–159) introduces multiple summaries using customer state and gender. Here we use product category and channel. Each grouping set creates its own aggregation grain.

```sql
SELECT p.product_type,s.channel,SUM(s.amount) AS revenue
FROM m13_sales s JOIN m13_products p USING(product_id)
GROUP BY p.product_type,s.channel ORDER BY 1,2;
```

```sql
SELECT p.product_type,s.channel,SUM(s.amount) AS revenue
FROM m13_sales s JOIN m13_products p USING(product_id)
GROUP BY ROLLUP(p.product_type,s.channel)
ORDER BY GROUPING(p.product_type,s.channel),1,2;
```

ROLLUP(a,b) expands to (a,b), (a), (). It does not include (b). Expected: 5 detail groups, 2 product subtotals, 1 grand total. Scooter revenue 570; Auto revenue 1080; grand total 1650.

```sql
SELECT p.product_type,s.channel,SUM(s.amount) AS revenue
FROM m13_sales s JOIN m13_products p USING(product_id)
GROUP BY CUBE(p.product_type,s.channel)
ORDER BY GROUPING(p.product_type,s.channel),1,2;
```

CUBE(a,b) includes (a,b), (a), (b), (). Expected 11 rows here, including 3 channel subtotals. For n independent dimensions CUBE produces 2^n grouping sets, not necessarily 2^n output rows.

```sql
SELECT p.product_type,s.channel,SUM(s.amount) AS revenue
FROM m13_sales s JOIN m13_products p USING(product_id)
GROUP BY GROUPING SETS ((p.product_type),(s.channel),())
ORDER BY GROUPING(p.product_type,s.channel),1,2;
```

Expected 6 rows. Explicit grouping sets are useful when stakeholders need only selected summaries, not all intersections. A single statement does not guarantee a particular number of physical scans; inspect EXPLAIN if performance matters.

## 2.2 Label subtotal rows without losing genuine NULLs

```sql
SELECT CASE WHEN GROUPING(p.product_type)=1 THEN 'ALL products'
            ELSE COALESCE(p.product_type,'Unknown product') END AS product_label,
       CASE WHEN GROUPING(s.channel)=1 THEN 'ALL channels'
            ELSE COALESCE(s.channel,'Unknown channel') END AS channel_label,
       GROUPING(p.product_type,s.channel) AS mask,
       COUNT(*) AS transactions,SUM(s.amount) AS revenue
FROM m13_sales s JOIN m13_products p USING(product_id)
GROUP BY CUBE(p.product_type,s.channel)
ORDER BY mask,product_label,channel_label;
```

| Mask | Grain |
|---:|---|
| 0 | Product and channel |
| 1 | Product only |
| 2 | Channel only |
| 3 | Grand total |

The rightmost argument is the least significant bit. The unknown-channel detail is $100. Do not use COALESCE alone to label totals. Do not sum the entire mixed-grain result: it counts facts once per grouping set.

## Part 2 knowledge check
Which grouping sets give region totals, category totals and grand total but no intersections? What is the grand-total mask for two dimensions?

**Check:** GROUPING SETS ((region),(category),()); mask 3.

### Example 13.3: Energy usage subtotal levels

**Goal and context.** The grouping mask distinguishes an unknown shift from a subtotal.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
SELECT site,shift,sum(kwh),grouping(site,shift) AS mask FROM (VALUES ('North','day',10),('North',NULL,5),('South','night',20)) t(site,shift,kwh) GROUP BY ROLLUP(site,shift) ORDER BY mask,site,shift;
```

**Expected result:** 3 detail rows, 2 site subtotals, grand total 35.

**Derived practice:** Replace ROLLUP with CUBE. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

Adds shift-only subtotals, including unknown shift.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

# Part 3: Distribution window functions
## 3.1 Ranking with ties
For ascending order, PERCENT_RANK = (rank−1)/(N−1), with 0 for a singleton; CUME_DIST = (preceding rows plus peers)/N. NTILE divides ordered rows into near-equal buckets, with larger buckets first. Descending order reverses the interpretation of preceding rows.

```sql
WITH scores(id,score) AS (VALUES (1,10),(2,20),(3,20),(4,40),(5,50))
SELECT id,score,RANK() OVER (ORDER BY score) AS r,
 ROUND(PERCENT_RANK() OVER (ORDER BY score)::numeric,2) AS pr,
 ROUND(CUME_DIST() OVER (ORDER BY score)::numeric,2) AS cd,
 NTILE(2) OVER (ORDER BY score,id) AS bucket
FROM scores ORDER BY score,id;
```

| Score | Rank | Percent rank | Cume dist | Bucket |
|---:|---:|---:|---:|---:|
| 10 | 1 | 0 | .20 | 1 |
| 20 | 2 | .25 | .60 | 1 |
| 20 | 2 | .25 | .60 | 1 |
| 40 | 4 | .75 | .80 | 2 |
| 50 | 5 | 1 | 1 | 2 |

PERCENT_RANK is not exactly the fraction strictly below (denominator N - 1). Highest tied values can have percent rank below 1. NTILE may split ties; use a stable ID for bucket reproducibility, but not for peer-preserving rank calculations.

```sql
WITH counts AS (
 SELECT c.customer_id,c.region,COUNT(s.sale_id) AS purchases
 FROM m13_customers c LEFT JOIN m13_sales s USING(customer_id)
 GROUP BY c.customer_id,c.region
)
SELECT *,NTILE(4) OVER (ORDER BY purchases DESC,customer_id) AS quartile,
 PERCENT_RANK() OVER (ORDER BY purchases DESC) AS pr_desc,
 CUME_DIST() OVER (ORDER BY purchases DESC) AS cd_desc
FROM counts ORDER BY purchases DESC,customer_id;
```

Expected: customer 1 has 2 purchases, customers 2 - 7 have 1, customer 8 has 0. Top bucket includes two customers. The six tied one-purchase customers share descending percent rank 1/7 and cume dist 7/8; they may occupy different buckets. With DESC, cume dist measures fraction at or above the current value.

```sql
SELECT customer_id,region,amount,
       CUME_DIST() OVER(PARTITION BY region ORDER BY amount) AS regional_position
FROM m13_sales JOIN m13_customers USING(customer_id)
ORDER BY region,amount,customer_id;
```

The partition changes the comparison population. The previous query ranks customers; this one ranks transactions within regions. These are different business questions. For percentile thresholds rather than row positions, use PERCENTILE_CONT or PERCENTILE_DISC.

```sql
SELECT percentile_cont(0.5) WITHIN GROUP (ORDER BY amount) AS median_sale FROM m13_sales;
```

Expected median: 190. This optional bridge revisits ordered-set aggregates in textbook Chapter 8.

### Example 13.4: Service latency distribution

**Goal and context.** Peers share rank but can cross bucket boundaries.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
SELECT ms, rank() OVER(ORDER BY ms),cume_dist() OVER(ORDER BY ms),ntile(2) OVER(ORDER BY ms,id) FROM (VALUES (1,10),(2,20),(3,20),(4,90)) t(id,ms) ORDER BY ms,id;
```

**Expected result:** ranks 1,2,2,4; cumulative .25,.75,.75,1; buckets 1,1,2,2.

**Derived practice:** Calculate median using percentile_cont(.5). Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

20.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

# Part 4: Statistical aggregates and interpretation
## 4.1 Population, sample and missingness
Chapter 13 distinguishes the target population from a sample and parameters from estimates. A complete database for this quarter is not automatically the population for future quarters. Define the unit of observation, eligible records, and target population before computing a statistic.

STDDEV and VARIANCE are aliases for sample versions (N−1); STDDEV_POP and VAR_POP use N. Standard deviation retains the original unit; variance has squared units. NULL values are ignored.

```sql
SELECT COUNT(*) AS rows,COUNT(x) AS nonnull_n,AVG(x) AS mean,
 VAR_SAMP(x) AS sample_variance,STDDEV_SAMP(x) AS sample_sd,
 VAR_POP(x) AS population_variance,STDDEV_POP(x) AS population_sd
FROM (VALUES (10::numeric),(20),(30),(NULL)) AS t(x);
```

Expected: rows 4, n 3, mean 20, sample variance 100, sample SD 10, population variance 66.6667, population SD 8.165. Sample SD is NULL for one usable observation. Zero SD describes no observed variation, not missing information.

```sql
SELECT product_type,COUNT(*) AS products,COUNT(base_msrp) AS priced_products,
 ROUND(AVG(base_msrp),2) AS average_msrp,
 ROUND(STDDEV_SAMP(base_msrp),2) AS sd_msrp,
 MIN(base_msrp),MAX(base_msrp)
FROM m13_products GROUP BY product_type ORDER BY product_type;
```

Auto: mean 350, sample SD 70.71, 2 priced of 3 products. Scooter: mean 150, sample SD 70.71, 2 priced of 2. Equal absolute dispersion is not equal relative dispersion. The 68% within one SD rule assumes approximately normal data; it is not universal.

### Example 13.5: Manufacturing variability

**Goal and context.** Sample and population denominators differ.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
SELECT count(x),avg(x),stddev_samp(x),stddev_pop(x) FROM (VALUES (8::numeric),(10),(12),(NULL)) t(x);
```

**Expected result:** n=3, mean=10, sample SD=2, population SD≈1.633.

**Derived practice:** Keep only one reading. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

Sample SD NULL; population SD 0.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

## 4.2 Covariance and Pearson correlation
Covariance depends on measurement units. Pearson r standardizes linear association; sign describes direction and absolute magnitude describes strength. A near-zero correlation does not rule out nonlinear relationships or confounding.

```sql
WITH pairs(x,y) AS (VALUES (1.0,2.0),(2.0,4.0),(3.0,6.0),(4.0,NULL))
SELECT COUNT(*) AS rows,REGR_COUNT(y,x) AS paired_n,
 CORR(y,x) AS r,COVAR_SAMP(y,x),COVAR_POP(y,x)
FROM pairs;
```

Expected: 4 rows, 3 pairs, r=1, sample covariance=2, population covariance=4/3. A constant variable makes correlation undefined (NULL). Report paired N, not merely COUNT(*).

```sql
SELECT CORR(x,y) AS linear_correlation
FROM (VALUES (-2,4),(-1,1),(0,0),(1,1),(2,4)) t(x,y);
```

Expected near zero despite exact relationship y=x². Correlation is not causation. A strong price correlation may reflect product mix rather than pricing policy.

### Example 13.6: Paired weather measurements

**Goal and context.** Incomplete pairs are ignored; correlation is not causation.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
SELECT regr_count(y,x),corr(y,x),covar_samp(y,x) FROM (VALUES (1::float8,5::float8),(2,3),(3,1),(4,NULL)) t(x,y);
```

**Expected result:** 3 pairs, correlation -1, covariance -2.

**Derived practice:** Make y constant. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

Correlation undefined (NULL).

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

## 4.3 Simple linear regression
The fitted line is y_hat = intercept + slope × x. PostgreSQL takes dependent variable Y first, explanatory variable X second. Slope has units Y per X; intercept predicts Y at X=0 and may have no practical meaning outside the observed range.

```sql
WITH pairs(x,y) AS (VALUES (1.0,3.0),(2.0,5.0),(3.0,7.0))
SELECT REGR_SLOPE(y,x) AS slope,REGR_INTERCEPT(y,x) AS intercept,
 REGR_R2(y,x) AS r_squared,REGR_COUNT(y,x) AS n,
 REGR_SLOPE(x,y) AS reversed_slope
FROM pairs;
```

Expected: slope 2, intercept 1, R² 1, n 3, reversed slope .5. R² describes in-sample fit, not causal attribution or future accuracy. A slope below one does not prove discounts: the intercept and residuals matter.

```sql
SELECT REGR_COUNT(s.amount,p.base_msrp) AS paired_n,
 ROUND(CORR(s.amount,p.base_msrp)::numeric,4) AS r,
 ROUND(REGR_SLOPE(s.amount,p.base_msrp)::numeric,4) AS slope,
 ROUND(REGR_INTERCEPT(s.amount,p.base_msrp)::numeric,2) AS intercept,
 ROUND(REGR_R2(s.amount,p.base_msrp)::numeric,4) AS r_squared
FROM m13_sales s JOIN m13_products p USING(product_id);
```

Seven complete pairs remain. This is transaction-weighted: repeated products contribute repeated observations. For a product-weighted analysis first aggregate actual sales by product. Inspect outliers and group-specific relationships before reporting conclusions.

```sql
WITH fits AS (
 SELECT REGR_SLOPE(s.amount,p.base_msrp) AS slope,
 REGR_INTERCEPT(s.amount,p.base_msrp) AS intercept
 FROM m13_sales s JOIN m13_products p USING(product_id)
)
SELECT s.sale_id,p.base_msrp,s.amount,
 ROUND((f.intercept+f.slope*p.base_msrp)::numeric,2) AS fitted,
 ROUND((s.amount-f.intercept-f.slope*p.base_msrp)::numeric,2) AS residual
FROM m13_sales s JOIN m13_products p USING(product_id) CROSS JOIN fits f
WHERE p.base_msrp IS NOT NULL ORDER BY s.sale_id;
```

## 4.4 SQL versus Python/R
SQL can compute the descriptive inputs and implement formulas explicitly, but core PostgreSQL does not provide a full standard regression inference report or general t-test API. Use dedicated tools for uncertainty, hypothesis tests, residual diagnostics, multivariable modeling and validation. Do not equate a database aggregate with statistical significance. 

### Example 13.7: AI demand model baseline

**Goal and context.** A simple regression is a baseline for more complex AI models; evaluate on later unseen observations.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
WITH t(x,y) AS (VALUES (1::float8,7::float8),(2,10),(3,13)) SELECT regr_slope(y,x),regr_intercept(y,x),regr_r2(y,x),regr_intercept(y,x)+4*regr_slope(y,x) AS forecast FROM t;
```

**Expected result:** slope 3, intercept 4, R² 1, forecast 16.

**Derived practice:** If actual at x=4 is 19, find absolute error. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

3; perfect training fit did not guarantee prediction accuracy.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

# Part 5: Applied ZoomZoom business analytics
The following examples require the course database. Verify names and data types with the schema query above. Values depend on your import; no live results are claimed.

```sql
SELECT MIN(sales_transaction_date),MAX(sales_transaction_date) FROM sales;
```

```sql
WITH RECURSIVE bounds AS (
 SELECT date_trunc('month',MIN(sales_transaction_date))::date AS lo,
 date_trunc('month',MAX(sales_transaction_date))::date AS hi FROM sales
), months(m) AS (
 SELECT lo FROM bounds WHERE lo IS NOT NULL
 UNION ALL
 SELECT (m+INTERVAL '1 month')::date FROM months CROSS JOIN bounds WHERE m < hi
), monthly AS (
 SELECT date_trunc('month',sales_transaction_date)::date AS m,
 COUNT(*) AS transactions,SUM(sales_amount) AS revenue FROM sales GROUP BY 1
)
SELECT months.m,COALESCE(transactions,0) AS transactions,
 CASE WHEN transactions IS NULL THEN 0 ELSE revenue END AS revenue
FROM months LEFT JOIN monthly USING(m) ORDER BY months.m;
```

Question: are missing months genuinely inactive or missing from the source? Validate against the reporting period and data completeness. For timestamptz, choose the business timezone before truncation.

```sql
SELECT p.product_type,s.channel,GROUPING(p.product_type,s.channel) AS grain,
 COUNT(*) AS transactions,SUM(s.sales_amount) AS revenue
FROM sales s JOIN products p USING(product_id)
GROUP BY ROLLUP(p.product_type,s.channel)
ORDER BY grain,1,2;
```

Before accepting totals, compare joined row counts and revenue with the original sales table. Check uniqueness of product_id and unmatched fact keys. A grand total is meaningful only for the chosen join population.

```sql
WITH activity AS (
 SELECT c.customer_id,COUNT(s.customer_id) AS purchases
 FROM customers c LEFT JOIN sales s USING(customer_id)
 WHERE c.is_test_customer IS FALSE
 GROUP BY c.customer_id
)
SELECT customer_id,purchases,NTILE(4) OVER(ORDER BY purchases DESC,customer_id) AS bucket,
 CUME_DIST() OVER(ORDER BY purchases DESC) AS cd
FROM activity ORDER BY purchases DESC,customer_id;
```

```sql
SELECT p.product_type,REGR_COUNT(s.sales_amount,p.base_msrp) AS n,
 CORR(s.sales_amount,p.base_msrp) AS r,
 REGR_SLOPE(s.sales_amount,p.base_msrp) AS slope,
 REGR_R2(s.sales_amount,p.base_msrp) AS r2
FROM sales s JOIN products p USING(product_id)
GROUP BY p.product_type ORDER BY p.product_type;
```

Interpret each category separately. If MSRP is constant within a category, slope or correlation may be undefined. Report the missing or insufficient observations instead of replacing statistics with zero.

```sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT product_type,COUNT(*),AVG(base_msrp)
FROM products GROUP BY product_type;
```

Module 12 connection: record actual plan nodes, estimates, loops and execution time from your environment. Do not infer a speed improvement merely from shorter SQL.

### Example 13.8: Audit missing measurement coverage

**Goal and context.** Coverage and value completeness are separate checks.

**Predict → run → inspect.** Before running, predict the output. Copy the SQL into DBeaver (PostgreSQL 16); run temporary-table examples in one session. If interrupted, remove only the named temporary table before retrying.

```sql
WITH expected AS (SELECT generate_series(1,4) AS day), observed(day,val) AS (VALUES (1,10),(3,NULL),(4,20)) SELECT e.day,CASE WHEN o.day IS NULL THEN 'missing row' WHEN val IS NULL THEN 'missing reading' ELSE 'observed' END AS status FROM expected e LEFT JOIN observed o USING(day) ORDER BY e.day;
```

**Expected result:** observed, missing row, missing reading, observed.

**Derived practice:** Count observed days. Predict the changed result before running.

<details>
<summary>Reveal self-check</summary>

2.

</details>

**Transfer:** Explain which assumption would need checking before applying this technique to production data. Record the result and compare it with your prediction.

# Part 6: Common mistakes and AI-generated SQL critique
| Mistake | Correction |
|---|---|
| Unbounded recursion | Use finite termination and cycle protection |
| COUNT(*) after calendar LEFT JOIN | Count a non-null fact key |
| Label every NULL as total | Use GROUPING |
| Sum all CUBE rows | Select the appropriate grain |
| Treat NTILE as tie-preserving | Use stable tie-breaks or a tie-aware rule |
| Use COUNT(*) for regression N | Use REGR_COUNT(Y,X) |
| Interpret correlation as effect | Report association and limitations |

## Critique exercise 
An AI assistant says: “CUBE produces only hierarchical subtotals; descending CUME_DIST is the share of customers with fewer purchases; a regression slope below 1 proves all vehicles are discounted.” Identify each error and propose corrected language.

**Check:** CUBE includes all grouping combinations; descending cumulative distribution includes peers and preceding higher values; discounts require comparing actual prices against MSRP, not merely interpreting a fitted slope. The intercept and residuals matter.

# Part 7: Practice and project preparation
Practise on the synthetic tables first; then adapt to sqlda.
1. Generate January–June 2024 and retain all empty months.
2. Traverse only the East manager's descendants, including the manager.
3. Produce category and channel totals but omit intersections.
4. Rank customers within region using ascending CUME_DIST and deterministic NTILE(2).
5. Compare sample and population SD of transaction amounts; explain your target population.
6. Fit a price regression on complete pairs; report slope, intercept, R² and paired N. Explain one possible confounder.
7. Run EXPLAIN ANALYZE on one of your queries and annotate observed evidence.

## Validation checkpoints
- Calendar: six rows, zero activity February, May and June; total revenue 1650.
- East hierarchy: IDs 2,4,6 (inclusive).
- Grouping sets: six summary rows including one grand total.
- Regional rankings: four customers per region; preserve nonbuyers.
- Dispersion: sample SD exceeds population SD for nonconstant data with more than one observation.
- Regression: seven usable pairs; do not count the missing MSRP row.
- Plan: use actual output; execution time is environment dependent.

## Cleanup
Only remove temporary tables created for this lecture in the current connection.

```sql
DROP TABLE IF EXISTS pg_temp.m13_org, pg_temp.m13_sales, pg_temp.m13_products, pg_temp.m13_customers;
```

# Summary
- Recursive queries generate successive working sets until no new rows remain; use safe bounds and paths.
- ROLLUP summarizes a hierarchy, CUBE all combinations, GROUPING SETS selected combinations.
- GROUPING distinguishes subtotal placeholders from source NULLs.
- Distribution functions describe relative position and bucket membership; ordering and ties matter.
- Dispersion, correlation and regression require explicit grain, missingness handling and interpretation.
- SQL is often the right place for exploratory summaries, while dedicated statistical tools support richer inference and modeling.

Next: Module 14, SQL case studies and Python integration foundations.

## References
- Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for Data Analytics* (4th ed.). Packt. Chapter 13, pp. 251–271; Chapter 8, pp. 158–160; Chapter 9, pp. 165–183.
- ALY 6420: Introduction to Querying. Canvas Module 13.
- PostgreSQL Global Development Group. [7.8 WITH Queries](https://www.postgresql.org/docs/16/queries-with.html); [7.2.4 Grouping Sets, Cube, Rollup](https://www.postgresql.org/docs/16/queries-table-expressions.html#QUERIES-GROUPING-SETS); [9.21 Aggregate Functions](https://www.postgresql.org/docs/16/functions-aggregate.html); [9.22 Window Functions](https://www.postgresql.org/docs/16/functions-window.html).
- Supplementary reading assigned in Canvas: Winand, M. [WITH RECURSIVE (Top Level)](https://modern-sql.com/caniuse/with_recursive_(top_level)).